A. Predict the price of the Uber ride from a given pickup point to the agreed drop-off
location. Perform following tasks:
1. Pre-process the dataset.
2. Identify outliers.
3. Check the correlation.
4. Implement linear regression and ridge, Lasso regression models.
5. Evaluate the models and compare their respective scores like R2, RMSE, 

In [21]:
#run once, used to calculate distance between two coordinates
pip install numpy

SyntaxError: invalid syntax (532271730.py, line 2)

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error


import time
from datetime import datetime

import geopy.distance
from geopy.distance import geodesic
# Load dataset
df = pd.read_csv("./uber.csv")
df.head()

,Unnamed: 0,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,24238194,2015-05-07 19:52:06.0000003,7.5,2015-05-07 19:52:06 UTC,-73.999817,40.738354,-73.999512,40.723217,1
1,27835199,2009-07-17 20:04:56.0000002,7.7,2009-07-17 20:04:56 UTC,-73.994355,40.728225,-73.994710,40.750325,1
2,44984355,2009-08-24 21:45:00.00000061,12.9,2009-08-24 21:45:00 UTC,-74.005043,40.740770,-73.962565,40.772647,1
3,25894730,2009-06-26 08:22:21.0000001,5.3,2009-06-26 08:22:21 UTC,-73.976124,40.790844,-73.965316,40.803349,3
4,17610152,2014-08-28 17:47:00.000000188,16.0,2014-08-28 17:47:00 UTC,-73.925023,40.744085,-73.973082,40.761247,5


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Unnamed: 0         200000 non-null  int64  
 1   key                200000 non-null  str    
 2   fare_amount        200000 non-null  float64
 3   pickup_datetime    200000 non-null  str    
 4   pickup_longitude   200000 non-null  float64
 5   pickup_latitude    200000 non-null  float64
 6   dropoff_longitude  199999 non-null  float64
 7   dropoff_latitude   199999 non-null  float64
 8   passenger_count    200000 non-null  int64  
dtypes: float64(5), int64(2), str(2)
memory usage: 23.4 MB


In [11]:
df = df.drop(columns = ['Unnamed: 0', 'key'])
df = df[df["fare_amount"] > 0]
df.dropna(inplace=True)

df["pickup_datetime"] = pd.to_datetime(df['pickup_datetime'])

In [12]:
df["day"] = df["pickup_datetime"].dt.day
df["month"] = df["pickup_datetime"].dt.month
df["year"] = df["pickup_datetime"].dt.year
df.drop(columns = ['pickup_datetime'], inplace = True)
df.head()

,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,day,month,year
0,7.5,-73.999817,40.738354,-73.999512,40.723217,1,7,5,2015
1,7.7,-73.994355,40.728225,-73.994710,40.750325,1,17,7,2009
2,12.9,-74.005043,40.740770,-73.962565,40.772647,1,24,8,2009
3,5.3,-73.976124,40.790844,-73.965316,40.803349,3,26,6,2009
4,16.0,-73.925023,40.744085,-73.973082,40.761247,5,28,8,2014


In [13]:
from pyproj import Geod
geod = Geod(ellps="WGS4")



ModuleNotFoundError: No module named 'pyproj'

In [51]:
Q1 = df["fare_amount"].quantile(0.25)
Q3 = df["fare_amount"].quantile(0.75)
IQR = Q3 - Q1

df = df[
    (df["fare_amount"] >= Q1 - 1.5 * IQR) &
    (df["fare_amount"] <= Q3 + 1.5 * IQR)
]

In [52]:
print("Correlation Matrix:")
print(df.corr(numeric_only=True))


Correlation Matrix:
                   fare_amount  pickup_longitude  pickup_latitude  \
fare_amount           1.000000          0.001254         0.003303   
pickup_longitude      0.001254          1.000000        -0.587628   
pickup_latitude       0.003303         -0.587628         1.000000   
dropoff_longitude    -0.003202          0.584119        -0.525871   
dropoff_latitude      0.000354         -0.535965         0.303413   
passenger_count       0.013387          0.003779        -0.005249   
distance              0.005095         -0.361844         0.646414   

                   dropoff_longitude  dropoff_latitude  passenger_count  \
fare_amount                -0.003202          0.000354         0.013387   
pickup_longitude            0.584119         -0.535965         0.003779   
pickup_latitude            -0.525871          0.303413        -0.005249   
dropoff_longitude           1.000000         -0.800242         0.003677   
dropoff_latitude           -0.800242          1.0000

In [59]:
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"])
df["hour"] = df["pickup_datetime"].dt.hour

X = df[
    [
        "distance",
        "passenger_count",
        'hour'
    ]
]

y = df["fare_amount"]

In [60]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [61]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso()
}

In [62]:
for name, model in models.items():

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    print("\n", name)
    print("R2 Score:", r2)
    print("RMSE:", rmse)


 Linear Regression
R2 Score: 4.9077071761383095e-05
RMSE: 4.132904974544038

 Ridge Regression
R2 Score: 4.907931495923279e-05
RMSE: 4.132904969908349

 Lasso Regression
R2 Score: -2.4605533867072538e-05
RMSE: 4.133057240815698


In [67]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error

# Load dataset
df = pd.read_csv("uber.csv")

# Remove unnecessary columns
df = df.drop(columns=["Unnamed: 0", "key"], errors="ignore")

# Remove missing values
df = df.dropna()

# Remove invalid fares and passenger counts
df = df[
    (df["fare_amount"] > 0) &
    (df["passenger_count"] > 0)
]

# Keep only reasonable NYC coordinates
df = df[
    df["pickup_longitude"].between(-74.3, -73.7) &
    df["dropoff_longitude"].between(-74.3, -73.7) &
    df["pickup_latitude"].between(40.5, 41.0) &
    df["dropoff_latitude"].between(40.5, 41.0)
]

# Convert datetime
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"])

# Extract pickup hour
df["hour"] = df["pickup_datetime"].dt.hour

# Calculate approximate distance in km
df["distance"] = np.sqrt(
    ((df["dropoff_latitude"] - df["pickup_latitude"]) * 111) ** 2 +
    ((df["dropoff_longitude"] - df["pickup_longitude"]) * 85) ** 2
) * 1.3

# Remove unrealistic distances
df = df[
    (df["distance"] > 0) &
    (df["distance"] < 50)
]

# Remove fare outliers using IQR
Q1 = df["fare_amount"].quantile(0.25)
Q3 = df["fare_amount"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df = df[
    (df["fare_amount"] >= lower) &
    (df["fare_amount"] <= upper)
]

# Correlation
print("Correlation with Fare:")
print(
    df[["distance", "passenger_count", "hour", "fare_amount"]]
    .corr()
)

# Features and target
X = df[["distance", "passenger_count", "hour"]]
y = df["fare_amount"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
# Standardization
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Linear Regression
linear = LinearRegression()
linear.fit(X_train, y_train)

pred_linear = linear.predict(X_test)

print("\nLinear Regression")
print("R2 Score:", r2_score(y_test, pred_linear))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_linear)))

# Ridge Regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

pred_ridge = ridge.predict(X_test)

print("\nRidge Regression")
print("R2 Score:", r2_score(y_test, pred_ridge))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_ridge)))

# Lasso Regression
lasso = Lasso(alpha=0.01)
lasso.fit(X_train, y_train)

pred_lasso = lasso.predict(X_test)

print("\nLasso Regression")
print("R2 Score:", r2_score(y_test, pred_lasso))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_lasso)))

Correlation with Fare:
                 distance  passenger_count      hour  fare_amount
distance         1.000000         0.000451 -0.037689     0.804745
passenger_count  0.000451         1.000000  0.013640     0.012673
hour            -0.037689         0.013640  1.000000    -0.013046
fare_amount      0.804745         0.012673 -0.013046     1.000000
Training rows: 141184
Testing rows: 35297

Linear Regression
R2 Score: 0.6514077091382537
RMSE: 2.445009820740798

Ridge Regression
R2 Score: 0.6514076664179169
RMSE: 2.4450099705600032

Lasso Regression
R2 Score: 0.6513716381111686
RMSE: 2.445136317706452
